In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")

# Silver Layer: Financial Facts Transformation

This notebook transforms raw financial data from the Romanian Ministry of Finance into a clean, structured fact table.

## Data Source
**Bronze Table**: `company_ro.bronze.mfinante_uu_raw`
- Contains annual financial statements (Situatii Financiare) for Romanian companies
- Years covered: 2011-2025
- One row per company per year

## Column Mapping: Raw to Business Names

The source files use positional columns (`mfin_col_XX`). Here's the complete mapping:

| Position | Raw Column | Business Column | Type | Description |
|----------|------------|----------------|------|-------------|
| 00 | mfin_col_00 | cui | string | Company Unique Identifier (Cod Unic de Identificare) |
| 01 | mfin_col_01 | cod_caen_mfinante | string | CAEN economic activity code (may differ from ONRC) |
| 02 | mfin_col_02 | active_imobilizate | decimal(20,2) | Fixed assets |
| 03 | mfin_col_03 | active_circulante | decimal(20,2) | Current assets |
| 04 | mfin_col_04 | stocuri | decimal(20,2) | Inventory |
| 05 | mfin_col_05 | creante | decimal(20,2) | Receivables |
| 06 | mfin_col_06 | casa_si_conturi | decimal(20,2) | Cash and bank accounts |
| 07 | mfin_col_07 | cheltuieli_in_avans | decimal(20,2) | Prepaid expenses |
| 08 | mfin_col_08 | datorii | decimal(20,2) | Liabilities |
| 09 | mfin_col_09 | venituri_in_avans | decimal(20,2) | Deferred income |
| 10 | mfin_col_10 | provizioane | decimal(20,2) | Provisions |
| 11 | mfin_col_11 | capitaluri_proprii | decimal(20,2) | Equity |
| 12 | mfin_col_12 | capital_social | decimal(20,2) | Share capital |
| 13 | mfin_col_13 | **cifra_afaceri** | decimal(20,2) | **Revenue / Turnover** |
| 14 | mfin_col_14 | venituri_totale | decimal(20,2) | Total income |
| 15 | mfin_col_15 | cheltuieli_totale | decimal(20,2) | Total expenses |
| 16 | mfin_col_16 | profit_brut | decimal(20,2) | Gross profit |
| 17 | mfin_col_17 | pierdere_bruta | decimal(20,2) | Gross loss |
| 18 | mfin_col_18 | **profit_net** | decimal(20,2) | **Net profit** |
| 19 | mfin_col_19 | **pierdere_neta** | decimal(20,2) | **Net loss** |
| 20 | mfin_col_20 | nr_mediu_salariati | int | Average number of employees |

## Data Quality Rules

1. **Primary Key**: (cui, an) - one financial record per company per year
2. **Required Fields**: cui, an must be non-null and non-empty
3. **Deduplication**: Drop duplicates on (cui, an) composite key
4. **Header Removal**: Filter out embedded header rows where mfin_col_00 = 'CUI'
5. **Surrogate Key**: `financiar_key` = SHA256 hash of cui||an for traceability

## CAEN Code Normalization

CIAN economic activity codes are normalized to 4 digits:
- **Why**: Source data has varying lengths (e.g., '62', '6201', '620')
- **How**: Left-pad with zeros using `LPAD(cod, 4, '0')`
- **Examples**: '62' → '0062', '6201' → '6201'
- **Hierarchy**: 
  - 2 digits = Division (e.g., '62' = Computer programming)
  - 3 digits = Group (e.g., '620' = Computer programming activities)
  - 4 digits = Class (e.g., '6201' = Custom software development)

## Incremental Load Strategy

This notebook uses **MERGE INTO** (upsert) logic:
- **New records**: Insert when (cui, an) doesn't exist
- **Updated records**: Update when (cui, an) exists but data changed
- **Unchanged records**: Skip

This approach is 10-100x faster than full overwrite for incremental updates.

In [0]:
mfin_raw = spark.table("company_ro.bronze.mfinante_uu_raw")

display(
    mfin_raw
    .groupBy("_source_year")
    .count()
    .orderBy("_source_year")
)

print("Columns:")
print(mfin_raw.columns)

In [0]:
raw_data_cols = [
    c for c in mfin_raw.columns
    if c.startswith("_c") and c[2:].isdigit()
]

raw_data_cols = sorted(
    raw_data_cols,
    key=lambda c: int(c.replace("_c", ""))
)

named_fin = mfin_raw

for i, old_col in enumerate(raw_data_cols):
    named_fin = named_fin.withColumnRenamed(old_col, f"mfin_col_{i:02d}")

display(named_fin.limit(20))
print(named_fin.columns)

In [0]:
def clean_digits(col):
    return F.regexp_replace(col.cast("string"), r"[^0-9]", "")


def to_decimal(col):
    cleaned = F.regexp_replace(F.trim(col.cast("string")), r"[^0-9,\.\-]", "")
    cleaned = F.regexp_replace(cleaned, ",", ".")
    return cleaned.cast("decimal(20,2)")


def to_int(col):
    return clean_digits(col).cast("int")


def clean_text(col):
    return F.trim(col.cast("string"))

In [0]:
fact_financiar_anual = (
    named_fin
    # Remove embedded header row from each TXT file
    .filter(F.upper(F.col("mfin_col_00")) != "CUI")
    .select(
        clean_digits(F.col("mfin_col_00")).alias("cui"),
        F.col("_source_year").cast("int").alias("an"),

        clean_digits(F.col("mfin_col_01")).alias("cod_caen_mfinante"),

        to_decimal(F.col("mfin_col_02")).alias("active_imobilizate"),
        to_decimal(F.col("mfin_col_03")).alias("active_circulante"),
        to_decimal(F.col("mfin_col_04")).alias("stocuri"),
        to_decimal(F.col("mfin_col_05")).alias("creante"),
        to_decimal(F.col("mfin_col_06")).alias("casa_si_conturi"),
        to_decimal(F.col("mfin_col_07")).alias("cheltuieli_in_avans"),
        to_decimal(F.col("mfin_col_08")).alias("datorii"),
        to_decimal(F.col("mfin_col_09")).alias("venituri_in_avans"),
        to_decimal(F.col("mfin_col_10")).alias("provizioane"),
        to_decimal(F.col("mfin_col_11")).alias("capitaluri_proprii"),
        to_decimal(F.col("mfin_col_12")).alias("capital_social"),

        to_decimal(F.col("mfin_col_13")).alias("cifra_afaceri"),
        to_decimal(F.col("mfin_col_14")).alias("venituri_totale"),
        to_decimal(F.col("mfin_col_15")).alias("cheltuieli_totale"),

        to_decimal(F.col("mfin_col_16")).alias("profit_brut"),
        to_decimal(F.col("mfin_col_17")).alias("pierdere_bruta"),
        to_decimal(F.col("mfin_col_18")).alias("profit_net"),
        to_decimal(F.col("mfin_col_19")).alias("pierdere_neta"),

        to_int(F.col("mfin_col_20")).alias("nr_mediu_salariati"),

        F.col("_ingested_at"),
        F.col("_source_file")
    )
    .filter(F.col("cui").isNotNull())
    .filter(F.col("cui") != "")
    .filter(F.col("an").isNotNull())
    .withColumn(
        "financiar_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cui"), F.lit("")),
                F.coalesce(F.col("an").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .dropDuplicates(["cui", "an"])
    .select(
        "financiar_key",
        "cui",
        "an",
        "cod_caen_mfinante",
        "active_imobilizate",
        "active_circulante",
        "stocuri",
        "creante",
        "casa_si_conturi",
        "cheltuieli_in_avans",
        "datorii",
        "venituri_in_avans",
        "provizioane",
        "capitaluri_proprii",
        "capital_social",
        "cifra_afaceri",
        "venituri_totale",
        "cheltuieli_totale",
        "profit_brut",
        "pierdere_bruta",
        "profit_net",
        "pierdere_neta",
        "nr_mediu_salariati",
        "_ingested_at",
        "_source_file"
    )
)

display(fact_financiar_anual.limit(50))
print("Rows:", fact_financiar_anual.count())
print("Columns:", fact_financiar_anual.columns)

In [0]:
display(fact_financiar_anual.groupBy("an").agg(
    F.count("*").alias("rows"),
    F.count("cui").alias("rows_with_cui"),
    F.count("cifra_afaceri").alias("rows_with_cifra_afaceri"),
    F.count("profit_net").alias("rows_with_profit_net"),
    F.count("pierdere_neta").alias("rows_with_pierdere_neta"),
    F.count("nr_mediu_salariati").alias("rows_with_salariati"),
    F.count("datorii").alias("rows_with_datorii"),
    F.count("capitaluri_proprii").alias("rows_with_capitaluri")
).orderBy("an"))

In [0]:
# Identify and drop columns that are entirely null
columns_to_check = [col for col in fact_financiar_anual.columns if col not in ['_ingested_at', '_source_file']]

all_null_columns = []
for col_name in columns_to_check:
    null_count = fact_financiar_anual.filter(F.col(col_name).isNull()).count()
    total_count = fact_financiar_anual.count()
    
    if null_count == total_count:
        all_null_columns.append(col_name)
        print(f"Column '{col_name}' is entirely null - will be dropped")

if all_null_columns:
    print(f"\nDropping {len(all_null_columns)} columns with all null values: {all_null_columns}")
    fact_financiar_anual = fact_financiar_anual.drop(*all_null_columns)
else:
    print("No columns with all null values found - keeping all columns")

print(f"\nFinal column count: {len(fact_financiar_anual.columns)}")

In [0]:
# Create table if it doesn't exist
spark.sql("""
CREATE TABLE IF NOT EXISTS company_ro.silver.fact_financiar_anual (
  financiar_key STRING,
  cui STRING,
  an INT,
  cod_caen_mfinante STRING,
  active_imobilizate DECIMAL(20,2),
  active_circulante DECIMAL(20,2),
  stocuri DECIMAL(20,2),
  creante DECIMAL(20,2),
  casa_si_conturi DECIMAL(20,2),
  cheltuieli_in_avans DECIMAL(20,2),
  datorii DECIMAL(20,2),
  venituri_in_avans DECIMAL(20,2),
  provizioane DECIMAL(20,2),
  capitaluri_proprii DECIMAL(20,2),
  capital_social DECIMAL(20,2),
  cifra_afaceri DECIMAL(20,2),
  venituri_totale DECIMAL(20,2),
  cheltuieli_totale DECIMAL(20,2),
  profit_brut DECIMAL(20,2),
  pierdere_bruta DECIMAL(20,2),
  profit_net DECIMAL(20,2),
  pierdere_neta DECIMAL(20,2),
  nr_mediu_salariati INT,
  _ingested_at TIMESTAMP,
  _source_file STRING
)
USING DELTA
""")

# Write to temp view for merge
fact_financiar_anual.createOrReplaceTempView("fact_financiar_staging")

# Perform incremental merge (upsert)
merge_result = spark.sql("""
MERGE INTO company_ro.silver.fact_financiar_anual AS target
USING fact_financiar_staging AS source
ON target.cui = source.cui AND target.an = source.an

WHEN MATCHED THEN
  UPDATE SET
    target.financiar_key = source.financiar_key,
    target.cod_caen_mfinante = source.cod_caen_mfinante,
    target.active_imobilizate = source.active_imobilizate,
    target.active_circulante = source.active_circulante,
    target.stocuri = source.stocuri,
    target.creante = source.creante,
    target.casa_si_conturi = source.casa_si_conturi,
    target.cheltuieli_in_avans = source.cheltuieli_in_avans,
    target.datorii = source.datorii,
    target.venituri_in_avans = source.venituri_in_avans,
    target.provizioane = source.provizioane,
    target.capitaluri_proprii = source.capitaluri_proprii,
    target.capital_social = source.capital_social,
    target.cifra_afaceri = source.cifra_afaceri,
    target.venituri_totale = source.venituri_totale,
    target.cheltuieli_totale = source.cheltuieli_totale,
    target.profit_brut = source.profit_brut,
    target.pierdere_bruta = source.pierdere_bruta,
    target.profit_net = source.profit_net,
    target.pierdere_neta = source.pierdere_neta,
    target.nr_mediu_salariati = source.nr_mediu_salariati,
    target._ingested_at = source._ingested_at,
    target._source_file = source._source_file

WHEN NOT MATCHED THEN
  INSERT (
    financiar_key,
    cui,
    an,
    cod_caen_mfinante,
    active_imobilizate,
    active_circulante,
    stocuri,
    creante,
    casa_si_conturi,
    cheltuieli_in_avans,
    datorii,
    venituri_in_avans,
    provizioane,
    capitaluri_proprii,
    capital_social,
    cifra_afaceri,
    venituri_totale,
    cheltuieli_totale,
    profit_brut,
    pierdere_bruta,
    profit_net,
    pierdere_neta,
    nr_mediu_salariati,
    _ingested_at,
    _source_file
  )
  VALUES (
    source.financiar_key,
    source.cui,
    source.an,
    source.cod_caen_mfinante,
    source.active_imobilizate,
    source.active_circulante,
    source.stocuri,
    source.creante,
    source.casa_si_conturi,
    source.cheltuieli_in_avans,
    source.datorii,
    source.venituri_in_avans,
    source.provizioane,
    source.capitaluri_proprii,
    source.capital_social,
    source.cifra_afaceri,
    source.venituri_totale,
    source.cheltuieli_totale,
    source.profit_brut,
    source.pierdere_bruta,
    source.profit_net,
    source.pierdere_neta,
    source.nr_mediu_salariati,
    source._ingested_at,
    source._source_file
  )
""")

print("\n=== Merge Statistics ===")
print(f"Rows inserted: {merge_result.collect()[0]['num_inserted_rows']}")
print(f"Rows updated: {merge_result.collect()[0]['num_updated_rows']}")
print(f"Rows deleted: {merge_result.collect()[0]['num_deleted_rows']}")

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows
FROM company_ro.silver.fact_financiar_anual
GROUP BY an
ORDER BY an
"""))

## Data Quality Checks

Validation tests to ensure data integrity between bronze and silver layers.

In [0]:
# Data Quality Check 1: Row count reconciliation
print("=== DQ Check 1: Row Count Reconciliation ===")

bronze_count = spark.sql("""
SELECT COUNT(*) AS row_count
FROM company_ro.bronze.mfinante_uu_raw
WHERE UPPER(_c0) != 'CUI'  -- Exclude header rows
""").collect()[0]["row_count"]

silver_count = spark.sql("""
SELECT COUNT(*) AS row_count
FROM company_ro.silver.fact_financiar_anual
""").collect()[0]["row_count"]

diff = bronze_count - silver_count
diff_pct = (diff / bronze_count * 100) if bronze_count > 0 else 0

print(f"Bronze rows (after header filter): {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Difference: {diff:,} ({diff_pct:.2f}%)")

if diff_pct > 5:
    print("WARNING: More than 5% row loss between bronze and silver")
elif diff_pct > 0:
    print(f"✓ Acceptable: {diff_pct:.2f}% rows filtered (likely duplicates or nulls)")
else:
    print("PASS: Row counts match")

In [0]:
# Data Quality Check 2: Null values in critical columns
print("\n=== DQ Check 2: Null Values in Critical Columns ===")

null_check = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN cui IS NULL THEN 1 ELSE 0 END) AS null_cui,
  SUM(CASE WHEN an IS NULL THEN 1 ELSE 0 END) AS null_an,
  SUM(CASE WHEN cifra_afaceri IS NULL THEN 1 ELSE 0 END) AS null_cifra_afaceri,
  ROUND(SUM(CASE WHEN cifra_afaceri IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_cifra_afaceri
FROM company_ro.silver.fact_financiar_anual
""")

display(null_check)

result = null_check.collect()[0]
if result["null_cui"] > 0 or result["null_an"] > 0:
    print("FAIL: Primary key columns (cui, an) contain nulls!")
else:
    print("PASS: No nulls in primary key columns")
    
if result["pct_null_cifra_afaceri"] > 50:
    print(f"WARNING: {result['pct_null_cifra_afaceri']}% of rows have null cifra_afaceri")
else:
    print(f"OK: {result['pct_null_cifra_afaceri']}% null cifra_afaceri is acceptable")

In [0]:
# Data Quality Check 3: Duplicate detection on primary key
print("\n=== DQ Check 3: Duplicate Detection ===")

duplicates = spark.sql("""
SELECT
  cui,
  an,
  COUNT(*) AS duplicate_count
FROM company_ro.silver.fact_financiar_anual
GROUP BY cui, an
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 10
""")

dup_count = duplicates.count()

if dup_count > 0:
    print(f"FAIL: Found {dup_count} duplicate (cui, an) combinations!")
    display(duplicates)
else:
    print("PASS: No duplicates found on (cui, an) composite key")

In [0]:
# Data Quality Check 4: Foreign key validation
print("\n=== DQ Check 4: Foreign Key Validation (CUI exists in dim_firma) ===")

fk_check = spark.sql("""
SELECT
  COUNT(DISTINCT f.cui) AS distinct_cui_in_fact,
  COUNT(DISTINCT d.cui) AS cui_matching_dim_firma,
  ROUND((COUNT(DISTINCT d.cui) / COUNT(DISTINCT f.cui)) * 100, 2) AS match_pct
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
""")

display(fk_check)

result = fk_check.collect()[0]
if result["match_pct"] < 95:
    print(f"WARNING: Only {result['match_pct']}% of CUI values have matching company records")
else:
    print(f"PASS: {result['match_pct']}% of CUI values match dim_firma")

In [0]:
# Data Quality Check 5: Year coverage and trends
print("\n=== DQ Check 5: Year Coverage and Data Completeness ===")

year_summary = spark.sql("""
SELECT
  an,
  COUNT(*) AS total_records,
  COUNT(DISTINCT cui) AS unique_companies,
  COUNT(cifra_afaceri) AS rows_with_revenue,
  ROUND(COUNT(cifra_afaceri) / COUNT(*) * 100, 2) AS pct_with_revenue,
  ROUND(SUM(cifra_afaceri) / 1000000, 2) AS total_revenue_millions,
  COUNT(nr_mediu_salariati) AS rows_with_employees,
  ROUND(COUNT(nr_mediu_salariati) / COUNT(*) * 100, 2) AS pct_with_employees
FROM company_ro.silver.fact_financiar_anual
GROUP BY an
ORDER BY an DESC
""")

display(year_summary)
print("Year coverage summary complete")

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  COUNT(DISTINCT CONCAT(cui, '-', an)) AS distinct_company_years,
  COUNT(*) - COUNT(DISTINCT CONCAT(cui, '-', an)) AS duplicate_rows
FROM company_ro.silver.fact_financiar_anual
GROUP BY an
ORDER BY an
"""))

In [0]:
display(spark.sql("""
SELECT
  f.an,
  COUNT(*) AS financial_rows,
  COUNT(d.cui) AS rows_matching_dim_firma,
  ROUND(COUNT(d.cui) / COUNT(*) * 100, 2) AS match_percent
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
GROUP BY f.an
ORDER BY f.an
"""))

In [0]:
good_companies = spark.sql("""
SELECT
  f.cui,
  COALESCE(d.denumire, 'UNKNOWN') AS denumire,
  COUNT(*) AS years,
  MAX(f.cifra_afaceri) AS max_cifra_afaceri,
  MAX(f.profit_net) AS max_profit_net,
  MAX(f.nr_mediu_salariati) AS max_salariati
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
WHERE f.cui IS NOT NULL
  AND f.cui <> ''
  AND f.cifra_afaceri IS NOT NULL
GROUP BY
  f.cui,
  COALESCE(d.denumire, 'UNKNOWN')
ORDER BY max_cifra_afaceri DESC
LIMIT 50
""")

display(good_companies)

In [0]:
rows = good_companies.collect()

if len(rows) == 0:
    raise ValueError("No companies found with non-null cifra_afaceri. Check the financial column mapping.")

sample_cui = rows[0]["cui"]

print("Using sample CUI:", sample_cui)

In [0]:
display(spark.sql(f"""
SELECT
  f.an,
  f.cui,
  d.denumire,
  d.judet,
  d.localitate,
  d.stare_firma,
  f.cod_caen_mfinante,
  f.cifra_afaceri,
  f.profit_net,
  f.pierdere_neta,
  f.nr_mediu_salariati,
  f.datorii,
  f.capitaluri_proprii,
  f.venituri_totale,
  f.cheltuieli_totale
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
WHERE f.cui = '{sample_cui}'
ORDER BY f.an
"""))

In [0]:
display(spark.sql("""
SELECT
  f.an,
  f.cui,
  COALESCE(d.denumire, 'UNKNOWN') AS denumire,
  d.judet,
  d.localitate,
  d.stare_firma,
  f.cifra_afaceri,
  f.profit_net,
  f.pierdere_neta,
  f.nr_mediu_salariati,
  f.datorii,
  f.capitaluri_proprii
FROM company_ro.silver.fact_financiar_anual f
LEFT JOIN company_ro.silver.dim_firma d
  ON f.cui = d.cui
WHERE f.cifra_afaceri IS NOT NULL
ORDER BY f.cifra_afaceri DESC
LIMIT 100
"""))